# Computational Set 5: Vibronic Spectra of a Diatomic: Absorption, Emission, and the Stokes Shift

### A single-mode Holstein model for MgH$^+$, parameterized from quantum chemistry

**Learning outcomes**

By the end of this notebook you will be able to:

1. Parameterize a two-level-plus-boson (Holstein) model from *ab initio* data: the electronic transition energy, the vibrational frequency, and the vibronic coupling (Huang--Rhys factor).
2. Select and critically compare levels of theory (global-hybrid TDDFT, range-separated TDDFT, and a multireference reference) for an excited-state calculation.
3. Build the vibronic Hamiltonian, compute absorption and emission spectra, and explain the physical origin of the **Stokes shift**.
4. Reproduce the spectrum a second way, from a real-time dipole correlation function, reusing the RK4 propagator you built in the entanglement notebook.

---

**A note on the Design Recipe.** As in the spin-$\tfrac12$ notebook, every function you write follows the Design Recipe: **Header** (name, argument types, return type) $\to$ **Purpose** $\to$ **Examples** (which become your tests) $\to$ **Body** $\to$ **Test** $\to$ **Debug/Iterate**. Scaffolding lightens as the notebook proceeds.

> **Instructor note (remove for student version).** This is the *complete* version. Each student-authored function marks its Design-Recipe **Body** with a `# --- Body (students complete this) ---` banner; blank those regions to produce the student version. Parts B--D are fully decoupled from Psi4 and run with the example parameters, so the physics half works even before the quantum-chemistry half is executed.

## The physical picture: why a displaced oscillator gives a Stokes shift

We model one electronic transition of a molecule coupled to **one** vibrational mode. Two electronic states, ground $|g\rangle$ and excited $|e\rangle$, each carry a harmonic potential along a nuclear coordinate $q$. The key physics is that the excited-state potential minimum is **displaced** relative to the ground state.

$$H = (E_{00}+\lambda)\,|e\rangle\langle e| \;+\; \hbar\omega\, a^\dagger a \;+\; \sqrt{S}\,\hbar\omega\,|e\rangle\langle e|\,(a+a^\dagger).$$

The last term is a **diagonal (Holstein) coupling**: the boson couples to the electronic *population* $|e\rangle\langle e|$. It displaces the oscillator equilibrium in the excited state, and that displacement is the entire origin of the Stokes shift.

> **Contrast with the entanglement notebook.** There the coupling was $g(a\sigma_+ + a^\dagger\sigma_-)$ --- an *excitation-exchange* term that swaps quanta and produces Rabi/polariton splitting, but **no Stokes shift**. Same two ingredients (a two-level system and a boson); the *structure* of the coupling is what changes the physics.

The dimensionless **Huang--Rhys factor** $S = \lambda/\hbar\omega$ sets everything: the Franck--Condon intensities follow a Poisson law $e^{-S}S^n/n!$, the vertical absorption sits at $E_{00}+\lambda$, the vertical emission at $E_{00}-\lambda$, and the **first-moment Stokes shift is $2\lambda = 2S\hbar\omega$**.

In [1]:
import numpy as np
import numpy.linalg as la
import matplotlib.pyplot as plt
from math import factorial

# Psi4 is only needed for Part A. Parts B-D run without it.
try:
    import psi4
    HAVE_PSI4 = True
except ImportError:
    HAVE_PSI4 = False
    print("Psi4 not found -- Part A calculations will be skipped, but Parts B-D still run.")

hbar = 1.0  # atomic-style units throughout; energies are in units of the vibrational quantum in Parts B-D

Psi4 not found -- Part A calculations will be skipped, but Parts B-D still run.


# Part A --- Parameterizing MgH$^+$ with quantum chemistry

MgH$^+$ is a closed-shell diatomic cation with a bound $A{}^1\Sigma^+ \leftarrow X{}^1\Sigma^+$ transition in the UV. Because it is a **diatomic**, the single vibrational mode *is* the bond stretch, so we can read every model parameter directly off a 1-D potential-energy scan --- no normal-mode analysis required.

### Choosing a level of theory

We compute the excited state three ways, in increasing sophistication:

| Method | What it captures | What it misses | When you would need better |
|---|---|---|---|
| **B3LYP** (global hybrid TDDFT) | valence excitations, cheaply | long-range exchange; underestimates charge-transfer & Rydberg states | donor--acceptor / CT excitations, extended systems |
| **CAM-B3LYP / $\omega$B97X** (range-separated TDDFT) | correct long-range exchange | still single-reference; dynamic correlation approximate | strong CT/Rydberg character |
| **CASPT2 / MRCI** (multireference, *pre-tabulated here*) | static + dynamic correlation, bond breaking | expensive; active-space dependent | multireference regions, dissociation |

> **MgH$^+$ is a deliberately benign control case.** It is compact, single-reference near equilibrium, and has no meaningful charge-transfer character, so all three methods should agree closely *here*. **That agreement is necessary but not sufficient**: the skill is knowing which feature of a *harder* molecule would break each method. Do not read "they agreed" as "any method is always fine," that "a global hybrid equals a range-separated one," or that "CASSCF/CASPT2 is the truth." Watch what happens to the three curves as you stretch the bond toward dissociation --- the near-equilibrium agreement is region-dependent.

In [2]:
# --- Psi4 setup and molecule builder ---
if HAVE_PSI4:
    psi4.set_memory('2 GB')
    psi4.core.set_output_file('mgh.out', False)

def make_mgh(R):
    """Return a Psi4 molecule for MgH+ at bond length R (Angstrom): charge +1, singlet."""
    return psi4.geometry(f"""
    1 1
    Mg 0.0 0.0 0.0
    H  0.0 0.0 {R}
    units angstrom
    """)

### 🧩 Function 1 --- `scan_pes` (Design Recipe)

**Header.** `scan_pes(R_values: array, method: str, excited: bool=False) -> np.ndarray`
**Purpose.** Compute the electronic energy of MgH$^+$ at each bond length, on either the ground or the first excited state.
**Examples.** `scan_pes([1.6, 1.7, 1.8], 'b3lyp')` returns three ground-state energies with a minimum near $R_e$; the excited scan lies above it.
**Body / Test / Debug.** Because these are live *ab initio* calls, the "test" is a sanity plot: the ground curve should have a single minimum, and the excited curve should sit above it (and, we hope, also be bound).

> ⚠️ **Check the TDDFT call against your Psi4 version.** The excitation-energy API has changed across Psi4 releases; the form below is the modern `tdscf_excitations` interface. Adjust only the inner call if your environment differs.

In [3]:
def scan_pes(R_values, method, excited=False, n_states=3, state_index=0, basis='aug-cc-pvtz'):
    """
    Electronic energy of MgH+ along the bond coordinate.

    Arguments
    ---------
    R_values    : bond lengths in Angstrom
    method      : density functional or method string, e.g. 'b3lyp', 'cam-b3lyp'
    excited     : if True, return the excited-state total energy (ground + excitation)
    n_states    : number of excited states to solve for (TDDFT)
    state_index : which excited root to follow (0 = lowest)

    Returns
    -------
    E : np.ndarray of total energies (Hartree)
    """
    psi4.set_options({'basis': basis, 'reference': 'rks', 'tdscf_states': n_states})
    E = []
    # --- Body (students complete this) ---
    for R in R_values:
        make_mgh(R)
        e_gs, wfn = psi4.energy(method, return_wfn=True)
        if not excited:
            E.append(e_gs)
        else:
            res = psi4.procrouting.response.scf_response.tdscf_excitations(
                      wfn, states=n_states, triplets='NONE')
            exc = sorted(r['EXCITATION ENERGY'] for r in res)  # Hartree
            E.append(e_gs + exc[state_index])
    # -------------------------------------
    return np.array(E)

In [4]:
# Run the two live TDDFT tiers (skipped automatically if Psi4 is unavailable).
R = np.linspace(1.00, 3.5, 20)   # Angstrom

if HAVE_PSI4:
    Eg_b3lyp   = scan_pes(R, 'b3lyp')
    Ee_b3lyp   = scan_pes(R, 'b3lyp',     excited=True)
    Eg_cam     = scan_pes(R, 'cam-b3lyp')
    Ee_cam     = scan_pes(R, 'cam-b3lyp', excited=True)
else:
    Eg_b3lyp = Ee_b3lyp = Eg_cam = Ee_cam = None

# Pre-tabulated multireference reference (CASPT2/MRCI). REPLACE with your own computed values.
# Columns: R(Ang), E_ground(Hartree), E_excited(Hartree). Placeholder illustrative shape only.
ref_R  = R.copy()
ref_Eg = -200.20 + 0.5*3.0*(ref_R-1.65)**2/27.211           # placeholder ground well
ref_Ee = -200.20 + 3.6/27.211 + 0.5*2.7*(ref_R-1.80)**2/27.211  # placeholder displaced excited well

**🚧 Your task.** Plot all available potential-energy curves on one axis (energies in eV relative to the ground-state minimum). Then answer, *from the data*:

1. **Go/no-go checkpoint:** is the excited state you followed actually **bound** (a real minimum), and is it dipole-allowed ($\Sigma^+\!\to\!\Sigma^+$)? You do not always get to use excited state #1 --- if the lowest root is repulsive, follow the next bound one.
2. Near $R_e$, do the three methods agree? What happens as $R \to 2.2$ Å?

In [5]:
def to_eV(E, ref): return (E - np.min(ref)) * 27.211386

plt.figure(figsize=(7,5))
if HAVE_PSI4:
    plt.plot(R, to_eV(Eg_b3lyp, Eg_b3lyp), 'o-', label='X  B3LYP')
    plt.plot(R, to_eV(Ee_b3lyp, Eg_b3lyp), 's-', label='A  B3LYP (TDDFT)')
    plt.plot(R, to_eV(Eg_cam,   Eg_cam),   'o--', label='X  CAM-B3LYP')
    plt.plot(R, to_eV(Ee_cam,   Eg_cam),   's--', label='A  CAM-B3LYP (TDDFT)')

#plt.plot(ref_R, to_eV(ref_Eg, ref_Eg), 'k^-',  label='X  CASPT2 (ref, pre-tabulated)')
#plt.plot(ref_R, to_eV(ref_Ee, ref_Eg), 'kv-',  label='A  CASPT2 (ref, pre-tabulated)')
plt.xlabel('bond length R  (Angstrom)'); plt.ylabel('energy  (eV, rel. to X min)')
plt.title('MgH$^+$ potential-energy curves'); plt.legend(fontsize=8); plt.tight_layout(); plt.show()

/sessions/lucid-festive-planck/tmp/ipykernel_9/3270233524.py:13: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  plt.title('MgH$^+$ potential-energy curves'); plt.legend(fontsize=8); plt.tight_layout(); plt.show()


### 🧩 Function 2 --- `fit_harmonic` (Design Recipe)

**Header.** `fit_harmonic(R: array, E: array) -> (R_min, k, E_min)`
**Purpose.** Fit a parabola $E(R)\approx E_\text{min}+\tfrac12 k (R-R_\text{min})^2$ near the minimum of a PES, returning the equilibrium position, force constant, and minimum energy.
**Examples.** For a known parabola with $R_\text{min}=1.7,\ k=3.0,\ E_\text{min}=-2.0$, the fit recovers those three numbers. (This becomes the test.)

This is the same curve-fitting muscle you used for geometry optimization; here we reuse it to extract $R_e$, $\omega$, and $T_e$.

In [6]:
def fit_harmonic(R, E, window=None):
    """
    Least-squares parabola fit near the minimum of a 1-D PES.

    Arguments
    ---------
    R      : bond lengths
    E      : energies at those bond lengths
    window : optional (Rlo, Rhi) to restrict the fit to the near-equilibrium region

    Returns
    -------
    (R_min, k, E_min) : equilibrium position, force constant d2E/dR2, minimum energy
    """
    R = np.asarray(R); E = np.asarray(E)
    if window is not None:
        m = (R >= window[0]) & (R <= window[1]); R, E = R[m], E[m]
    # --- Body (students complete this) ---
    c2, c1, c0 = np.polyfit(R, E, 2)   # E = c2 R^2 + c1 R + c0
    R_min = -c1 / (2*c2)
    k     = 2*c2
    E_min = c0 + c1*R_min + c2*R_min**2
    # -------------------------------------
    return R_min, k, E_min

In [7]:
# Test: recover known parabola parameters
Rt = np.linspace(1.4, 2.0, 25)
Et = -2.0 + 0.5*3.0*(Rt-1.7)**2
R_min, k, E_min = fit_harmonic(Rt, Et)
assert np.isclose(R_min, 1.7, atol=1e-6)
assert np.isclose(k, 3.0, atol=1e-6)
assert np.isclose(E_min, -2.0, atol=1e-6)
print("✅ fit_harmonic tests passed!")

✅ fit_harmonic tests passed!


### 🧩 Function 3 --- `huang_rhys` (Design Recipe)

**Header.** `huang_rhys(dR: float, omega: float, mu: float) -> float`
**Purpose.** Compute the dimensionless Huang--Rhys factor for a diatomic from the bond-length displacement between the two electronic minima.
**Examples.** With $\Delta R = 0.15$ Å, $\tilde\nu = 1650$ cm$^{-1}$, $\mu = 0.968$ amu, $S \approx 0.53$ (verified value).

$$S = \frac{\mu\,\omega}{2\hbar}\,\Delta R^2 = \tfrac12\left(\Delta R\sqrt{\mu\omega/\hbar}\right)^2.$$

This is the single formula that turns quantum-chemistry geometry into vibronic coupling --- the heart of the parameterization.

In [8]:
HBAR_SI = 1.054571817e-34   # J s
AMU     = 1.66053907e-27    # kg
C_CM    = 2.99792458e10     # cm/s

def huang_rhys(dR_ang, nu_cm, mu_amu):
    """
    Huang-Rhys factor S for a diatomic from a PES scan.

    Arguments
    ---------
    dR_ang  : |R_e(excited) - R_e(ground)| in Angstrom
    nu_cm   : vibrational wavenumber in cm^-1
    mu_amu  : reduced mass in amu

    Returns
    -------
    S : dimensionless Huang-Rhys factor (= lambda / hbar*omega)

    Examples
    --------
    huang_rhys(0.15, 1650, 0.968) == 0.533   (approximately)
    """
    # --- Body (students complete this) ---
    omega = 2*np.pi*C_CM*nu_cm      # rad/s
    mu    = mu_amu*AMU
    dR    = dR_ang*1e-10
    S     = (mu*omega/(2*HBAR_SI))*dR**2
    # -------------------------------------
    return S

In [9]:
# Test against verified values
assert np.isclose(huang_rhys(0.15, 1650, 0.968), 0.533, atol=2e-3)
assert np.isclose(huang_rhys(0.20, 1650, 0.968), 0.947, atol=2e-3)
assert np.isclose(huang_rhys(0.00, 1650, 0.968), 0.0)
print("✅ huang_rhys tests passed!")

✅ huang_rhys tests passed!


### Extract the model parameters

From each method's curves: $R_e^X, R_e^A$ (giving $\Delta R$), $\omega$ from the ground-state curvature, and the adiabatic gap $E_{00}=E_\text{min}^A-E_\text{min}^X$. Feed $\Delta R,\ \omega,\ \mu$ into `huang_rhys`. Compare the three methods --- and note that the *near-equilibrium* parameters (which is all the spectrum needs) are far more robust than the long-$R$ tails.

> **Generalizing to polyatomics (note).** For a molecule with many modes you cannot use a single bond length. Instead: (i) do a normal-mode analysis at the ground minimum; (ii) get the excited-state gradient *at the ground geometry* (the vertical-gradient / linear-vibronic-coupling method) or the excited minimum; (iii) project the displacement onto each mode $k$ to get $S_k$, and sum: $\lambda = \sum_k \lambda_k = \sum_k S_k\hbar\omega_k$. The single-mode diatomic here is the clean special case where mode = bond stretch.

# Part B --- Building the single-mode Holstein model

We now assemble the vibronic Hamiltonian in the electronic$\otimes$vibrational basis, truncating the boson at $N$ levels. We reuse the tensor-product helper `kron` and the density-matrix machinery in the spirit of the earlier notebooks.

**Convention.** We parameterize by the **adiabatic** gap $E_{00}$ (minimum-to-minimum) and write the electronic origin as $E_{00}+\lambda$ so that, after the coupling displaces the excited well, its relaxed minimum sits exactly at $E_{00}$. This makes the vertical-vs-adiabatic distinction from Part A show up directly in the code.

### 🧩 Function 4 --- `holstein_hamiltonian` (Design Recipe)

**Header.** `holstein_hamiltonian(E00, omega, S, N) -> (2N+2)x(2N+2) array`
**Purpose.** Build the single-mode Holstein Hamiltonian in the $|el\rangle\otimes|vib\rangle$ basis.
**Examples.** With $S=0$ the two electronic blocks are undisplaced harmonic ladders (no coupling); $H$ is always Hermitian.

In [10]:
def holstein_hamiltonian(E00, omega, S, N, hbar=1.0):
    """
    Single-mode Holstein (displaced-oscillator) Hamiltonian.

    Arguments
    ---------
    E00   : adiabatic 0-0 gap (relaxed excited minimum sits here)
    omega : vibrational quantum
    S     : Huang-Rhys factor
    N     : boson truncation (vibrational levels 0..N)

    Returns
    -------
    H : Hermitian (2*(N+1)) x (2*(N+1)) matrix
    """
    a    = np.diag(np.sqrt(np.arange(1, N+1)), 1)
    adag = a.conj().T
    nop  = adag @ a
    Iv   = np.eye(N+1)
    Pe   = np.array([[0,0],[0,1]]); Iel = np.eye(2)
    lam  = S*hbar*omega
    g    = np.sqrt(S)
    # --- Body (students complete this) ---
    H = ((E00+lam)*np.kron(Pe, Iv)
         + hbar*omega*np.kron(Iel, nop)
         + g*hbar*omega*np.kron(Pe, (a + adag)))
    # -------------------------------------
    return H

In [11]:
# Tests
H = holstein_hamiltonian(E00=8.0, omega=1.0, S=0.9, N=30)
assert H.shape == (62, 62)
assert np.allclose(H, H.conj().T)                          # Hermitian
H0 = holstein_hamiltonian(8.0, 1.0, 0.0, 30)               # no coupling
assert np.allclose(H0, np.diag(np.diag(H0)))               # diagonal when S=0
print("✅ holstein_hamiltonian tests passed!")

✅ holstein_hamiltonian tests passed!


# Part C --- Absorption and emission by diagonalization

At $T=0$, absorption starts from $|g,0\rangle$ and emission from the *relaxed* excited vibrational ground state $|e,0'\rangle$ (Kasha's rule). Within the Condon approximation the dipole is a constant, so line intensities are Franck--Condon factors $|\langle \text{final}|\text{initial}\rangle|^2$, which for this model form a Poisson progression.

### 🧩 Function 5 --- `franck_condon_factors` (Design Recipe, light)

**Purpose.** Return the analytic $T=0$ Franck--Condon intensities $e^{-S}S^n/n!$ for $n=0\ldots n_\text{max}$.

In [12]:
def franck_condon_factors(S, n_max):
    """Poisson Franck-Condon intensities e^{-S} S^n / n! for n = 0..n_max."""
    n = np.arange(n_max+1)
    return np.exp(-S) * S**n / np.array([factorial(i) for i in n], dtype=float)

In [13]:
fc = franck_condon_factors(0.9, 5)
assert np.isclose(fc.sum(), 1.0, atol=1e-3)                # normalized
assert np.isclose(fc[0], np.exp(-0.9), atol=1e-6)
assert np.allclose(fc, [0.4066,0.3659,0.1647,0.0494,0.0111,0.0020], atol=1e-3)
print("✅ franck_condon_factors tests passed!")

✅ franck_condon_factors tests passed!


### 🧩 Function 6 --- `broaden_spectrum` (Design Recipe, light)

**Purpose.** Convert a stick spectrum (line positions + intensities) into a smooth lineshape by summing Gaussians of width $\sigma$ on an energy grid.

In [14]:
def broaden_spectrum(centers, intensities, grid, sigma):
    """Sum of Gaussians: smooth lineshape from a stick spectrum."""
    y = np.zeros_like(grid, dtype=float)
    for c, I in zip(centers, intensities):
        y += I*np.exp(-(grid-c)**2/(2*sigma**2))
    return y

In [15]:
grid = np.linspace(-5, 5, 400)
y = broaden_spectrum([0.0], [1.0], grid, 0.5)
assert np.isclose(grid[np.argmax(y)], 0.0, atol=0.03)      # peak at the line
area = np.sum(y)*(grid[1]-grid[0])                         # simple Riemann area (version-independent)
assert np.isclose(area, np.sqrt(2*np.pi)*0.5, rtol=1e-2)   # area = I*sigma*sqrt(2pi)
print("✅ broaden_spectrum tests passed!")

✅ broaden_spectrum tests passed!


### Compute both spectra and read off the Stokes shift

We diagonalize $H$ (reusing `la.eigh` from the spin-$\tfrac12$ notebook), project the Condon dipole onto the eigenstates for absorption (from $|g,0\rangle$) and emission (from $|e,0'\rangle$), broaden, and plot.

In [16]:
# --- model parameters: use Part A values if you computed them, else these examples ---
E00, omega, S, N = 8.0, 1.0, 0.9, 30       # energies in units of hbar*omega
lam = S*omega

H = holstein_hamiltonian(E00, omega, S, N)
E, V = la.eigh(H)

Iv = np.eye(N+1)
mu = np.kron(np.array([[0,1],[1,0]]), Iv)   # Condon dipole (couples g<->e, identity on vib)

# Absorption: initial |g,0>
g0 = np.zeros(2*(N+1)); g0[0] = 1.0
Ag = mu @ g0
absE, absI = [], []
for k in range(len(E)):
    w = abs(np.vdot(V[:,k], Ag))**2
    if w > 1e-9: absE.append(E[k]); absI.append(w)
absE, absI = np.array(absE), np.array(absI)

# Emission: relaxed excited vibrational ground state |e,0'>
a = np.diag(np.sqrt(np.arange(1,N+1)),1); adag=a.conj().T
He = (E00+lam)*Iv + omega*(adag@a) + np.sqrt(S)*omega*(a+adag)
ee, ve = la.eigh(He)
e0 = np.kron(np.array([0,1]), ve[:,0]); Ae = mu @ e0; Ee0 = ee[0]
emE, emI = [], []
for m in range(N+1):
    fin = np.kron(np.array([1,0]), np.eye(N+1)[m])
    w = abs(np.vdot(fin, Ae))**2
    if w > 1e-9: emE.append(Ee0 - m*omega); emI.append(w)
emE, emI = np.array(emE), np.array(emI)

# first moments and Stokes shift
abs_m = np.sum(absE*absI)/absI.sum()
em_m  = np.sum(emE*emI)/emI.sum()
print(f"absorption first moment = {abs_m:.3f}   (E00+lambda = {E00+lam})")
print(f"emission   first moment = {em_m:.3f}   (E00-lambda = {E00-lam})")
print(f"STOKES SHIFT = {abs_m-em_m:.3f}   (2*lambda = {2*lam})")

# verify numeric FC vs analytic Poisson (absorption)
o = np.argsort(absE)
print("numeric FC :", np.round(absI[o][:5], 4))
print("Poisson    :", np.round(franck_condon_factors(S,4), 4))

absorption first moment = 8.900   (E00+lambda = 8.9)
emission   first moment = 7.100   (E00-lambda = 7.1)
STOKES SHIFT = 1.800   (2*lambda = 1.8)
numeric FC : [0.4066 0.3659 0.1647 0.0494 0.0111]
Poisson    : [0.4066 0.3659 0.1647 0.0494 0.0111]


In [17]:
grid = np.linspace(E00-5, E00+6, 1500)
A  = broaden_spectrum(absE, absI, grid, 0.25)
Em = broaden_spectrum(emE, emI, grid, 0.25)

plt.figure(figsize=(8,4.5))
plt.plot(grid, A,  color='#2980b9', lw=2, label='absorption')
plt.fill_between(grid, A,  color='#2980b9', alpha=0.12)
plt.plot(grid, Em, color='#e67e22', lw=2, label='emission')
plt.fill_between(grid, Em, color='#e67e22', alpha=0.12)
plt.axvline(E00, color='k', ls=':', lw=1); plt.text(E00, A.max()*1.02, '0-0', ha='center')
plt.annotate('', xy=(abs_m, A.max()*0.5), xytext=(em_m, A.max()*0.5),
             arrowprops=dict(arrowstyle='<|-|>', color='green', lw=2))
plt.text((abs_m+em_m)/2, A.max()*0.55, r'Stokes shift $=2\lambda$', ha='center', color='green')
plt.xlabel('energy  (units of $\\hbar\\omega$)'); plt.ylabel('intensity')
plt.title('Absorption and emission: mirror images offset by $2\\lambda$')
plt.legend(); plt.tight_layout(); plt.show()

# Part D --- Capstone: the spectrum from a real-time dipole correlation function

The same absorption spectrum is the Fourier transform of the dipole autocorrelation function
$$C(t) = \langle \mu(t)\,\mu(0)\rangle = \mathrm{Tr}\!\big[\mu\, e^{-iHt/\hbar}\,\mu\rho_0\, e^{+iHt/\hbar}\big],\qquad I(\omega)\propto \mathrm{Re}\!\int_0^\infty\! e^{i\omega t} C(t)\,dt.$$

The object $A(t)=e^{-iHt/\hbar}(\mu\rho_0)e^{+iHt/\hbar}$ obeys exactly the Liouville--von Neumann equation $\dot A = -\tfrac{i}{\hbar}[H,A]$ --- so we can propagate it with the **very same `rk4_step` and `liouville_rhs` you wrote in the entanglement notebook**, then Fourier transform. This ties the two notebooks together: the dynamics tools built for entanglement also produce spectra.

In [18]:
# Reused from the entanglement notebook (Design Recipe functions you already wrote):
def commutator(A, B):
    return A @ B - B @ A

def liouville_rhs(H, rho, hbar=1.0):
    return -1j/hbar * commutator(H, rho)

def rk4_step(rho, H, dt, hbar=1.0):
    k1 = liouville_rhs(H, rho, hbar)
    k2 = liouville_rhs(H, rho + 0.5*dt*k1, hbar)
    k3 = liouville_rhs(H, rho + 0.5*dt*k2, hbar)
    k4 = liouville_rhs(H, rho + dt*k3, hbar)
    return rho + (dt/6.0)*(k1 + 2*k2 + 2*k3 + k4)

def density_matrix(ket):
    return ket @ ket.conj().T

### 🧩 Function 7 --- `dipole_correlation` (Design Recipe, capstone)

**Header.** `dipole_correlation(H, mu, rho0, dt, nt) -> (t, C)`
**Purpose.** Propagate $A(t)$ with RK4 and return the dipole autocorrelation $C(t)=\mathrm{Tr}[\mu A(t)]$.
**Examples / Test.** Integrated against the FFT below, the band it produces must land on the same progression as Part C.

In [19]:
def dipole_correlation(H, mu, rho0, dt, nt, hbar=1.0):
    """
    Real-time dipole autocorrelation C(t) = Tr[mu A(t)], with A(0) = mu @ rho0
    propagated under the Liouville-von Neumann equation via RK4.

    Returns
    -------
    (t, C) : time grid and complex correlation function
    """
    t = np.arange(nt)*dt
    C = np.zeros(nt, dtype=complex)
    # --- Body (students complete this) ---
    A = mu @ rho0
    for i in range(nt):
        C[i] = np.trace(mu @ A)
        A = rk4_step(A, H, dt, hbar)
    # -------------------------------------
    return t, C

In [20]:
rho0 = density_matrix(g0.reshape(-1,1))     # |g,0><g,0|
dt, nt = 0.01, 8000
t, C = dipole_correlation(H, mu, rho0, dt, nt)

gamma = 0.04                                  # phenomenological lifetime broadening
Cw = C*np.exp(-gamma*t)
w  = np.fft.fftshift(np.fft.fftfreq(nt, dt)*2*np.pi)
# absorption ~ Re integral e^{+i w t} C dt   ->  N * ifft   (note the sign convention!)
Iw = np.fft.fftshift(np.real(np.fft.ifft(Cw))*nt*dt)

mask = (w > E00-4) & (w < E00+6)
peak = w[mask][np.argmax(Iw[mask])]
strongest = absE[o][np.argmax(absI[o])]
print(f"correlation/FFT band peak      = {peak:.3f}")
print(f"diagonalization strongest line = {strongest:.3f}")
assert abs(peak - strongest) < 0.12
print("✅ correlation-function spectrum matches the diagonalization result!")

plt.figure(figsize=(8,4.5))
A_ref = broaden_spectrum(absE, absI, w, 0.15)
plt.plot(w[mask], (Iw[mask]/Iw[mask].max()), color='#8e44ad', lw=2, label='from $C(t)$ (RK4 + FFT)')
plt.plot(w[mask], (A_ref[mask]/A_ref[mask].max()), 'k--', lw=1.5, label='from diagonalization (Part C)')
plt.xlabel('energy  (units of $\\hbar\\omega$)'); plt.ylabel('absorption (norm.)')
plt.title('Two routes, one spectrum'); plt.legend(); plt.tight_layout(); plt.show()

correlation/FFT band peak      = 8.011
diagonalization strongest line = 8.000
✅ correlation-function spectrum matches the diagonalization result!


### Wrap-up and extension

You parameterized a vibronic model from *ab initio* data, compared three levels of theory on a benign control case, and computed absorption/emission spectra two independent ways --- static diagonalization and real-time RK4 dynamics --- obtaining the same Stokes shift $2\lambda = 2S\hbar\omega$.

> **Finite-temperature extension.** Replace the initial state $\rho_0=|g,0\rangle\langle g,0|$ with a thermal (Boltzmann) mixture over ground-state vibrational levels and rerun `dipole_correlation`. Hot bands appear, and the lineshape broadens --- the correlation-function route handles temperature naturally, whereas the $T=0$ Franck--Condon picture does not.